In [ ]:
F1_THRESHOLD = 0.0

import json
import pandas as pd
import os

columns = ["focal", "focal_factor", "depth", "filters", "kernel"]
experiments = ["Bin_SDSS", "Bin_DES", "Bin_H_1", "Bin_H_2", "Bin_H_3"]
setups = ["paper3_models_step2"]#, "paper3_models_same_padding"]

dataset = pd.DataFrame({}, columns= columns + ["best_f1"])

for experiment in experiments:
    for setup in setups:
        for i in range(300):
            path_base = f'{setup}/{experiment}/{i}'
            if not os.path.exists(path_base):
                continue
            with open(f'{path_base}/model.json', 'r') as mf:
                model_dict = json.load(mf)
            if os.path.exists(f'{path_base}/result.json'):
                with open(f'{path_base}/result.json', 'r') as rf:
                    res_dict = json.load(rf)
                model_dict["best_f1"] = res_dict["best_f1"]
            else:
                model_dict["best_f1"] = None
            model_dict["experiment"] = experiment
            model_dict["setup"] = setup
            data = pd.DataFrame(model_dict,  index = [i])
            dataset = pd.concat([dataset, data])

dataset = dataset[~dataset["best_f1"].isna()]
dataset = dataset[dataset["best_f1"] >= F1_THRESHOLD]
dataset["relative_f1"] = dataset.apply(lambda x: x["best_f1"] / dataset[dataset["experiment"] == x["experiment"]]["best_f1"].max(), axis=1)


In [ ]:
import matplotlib.pyplot as plt

TOP_N_VALUES = 10

fig, ax = plt.subplots(len(columns), len(experiments), figsize=(5 * len(experiments), 4 * len(columns)))


for i, experiment in enumerate(experiments):
    ex_data = dataset[dataset["experiment"] == experiment]
    
    for j, column in enumerate(columns):
        values = []
        max_values = []
        top_5_max_values = []
        top_n_values = []
        unique_parameter_vales = sorted(ex_data[column].unique())

        for value in unique_parameter_vales:
            parameter_values = ex_data[ex_data[column] == value]
            
            values.append(parameter_values["best_f1"])
            max_values.append(parameter_values["best_f1"].max())
            top_values = parameter_values.sort_values("best_f1").tail(TOP_N_VALUES)["best_f1"]
            top_n_values.append(top_values)
            top_5_max_values.append(top_values.mean())

        positions = range(1, len(unique_parameter_vales) + 1)
        # ax[j][i].boxplot(values, positions = positions, labels = unique_parameter_vales, showfliers=False)
        ax[j][i].boxplot(top_n_values, positions = positions, labels = unique_parameter_vales, showfliers=False)
        ax[j][i].plot(positions, max_values, color="red", label="Max F1 value", marker='o')
        ax[j][i].plot(positions, top_5_max_values, color="royalblue", label="Top 10 F1 values, avg", marker='o')
        ax[j][i].legend()
        ax[j][i].set_title(column + f"({experiment}, n = {len(ex_data)})")
        ax[j][i].set_xlabel(column)
        ax[j][i].set_ylabel("F1")
        ax[j][i].grid()
        # plt.show()
# plt.legend()
plt.tight_layout()
# for column in ["depth", "filters", "kernel", "dropout", "dense_depth", "dense"]:




In [ ]:
dataset[(dataset["experiment"] == "Bin_SDSS")].sort_values("best_f1").tail(10)

In [ ]:
dataset[(dataset["experiment"] == "Bin_DES")].sort_values("best_f1").tail(10)

In [ ]:
dataset[dataset["experiment"] == "Bin_H_1"].sort_values("best_f1").tail(10)

In [ ]:
dataset[dataset["experiment"] == "Bin_H_2"].sort_values("best_f1").tail(10)

In [ ]:
dataset[dataset["experiment"] == "Bin_H_3"].sort_values("best_f1").tail(10)